# Car ads annotation on 3 Jakobson's functions of language dimensions

We annotate the car ads in this way:

- `informativeness` using 5 point Likert scale
- `expressiveness` using 5 point Likert scale
- `phatic` using 5 point Likert scale
- `dominant_dimension` this is a classification that choose the most prevalent dimension
- `dominant_dimension_score` using 5 point Likert scale only the dimension that was chose before

Architecture:

1. Gemma 4 as `Gemma_annotater`
2. Mistral as `Mistral_annotater`
3. Router decides `accept`, `send_to_review`, or `ask_human`



## Theory Notes

They are based on Roman Jakobson functions of language in his 1960 "Closing Statement: Linguistics and Poetics,

- `informativeness`: referential or factual function, pointing to the real world and giving concrete information (Focused on the Context (the topic)) (Jakobson, 1960)
- `expressiveness`: emotive function, where the initiator expresses feeling, desire, admiration, aspiration, or emotional tone ( Focused on the addresser (sender)) (Jakobson, 1960)
- `phatic`:  relationship-maintaining function  or discontinue communication, mostly used in social interaction, where the ad creates or maintains an affective bond with the customer (Focused on the Contact (the channel)) (Jakobson, 1960)

Important design choice:

- we do **not** collapse informativeness and expressiveness into one single axis first;
- we score all three dimensions independently from 1 to 5;
- only after that do we derive the dominant dimension.

Reference

Jakobson, Roman. "Linguistics and Poetics". Volume III Poetry of Grammar and Grammar of Poetry, Berlin, Boston: De Gruyter Mouton, 1981, pp. 18-51. https://doi.org/10.1515/9783110802122.18


#Imports
They are specialy made for google colab if you are using something else please comment drive.mount function

In [2]:
from google.colab import drive #this is useful only if you want to access the documents from Google Drive
drive.mount('/content/drive', force_remount=False) #this connects it to your Google Drive to the colab runtime and force_remount is False since we do not want to remount again if Drive is connected

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gliner-Work.Dauphine") #my folder name
DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv" # / joins folder and file names
OUTPUT_DIR = PROJECT_ROOT / "communication_function_outputs" #this is the output file
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) #this is useful to avoid crashes- if file does not exist it creates it, if it exists it does not crash

os.chdir(PROJECT_ROOT) #this change the working director to my current working folder


#this is useful only if you want to add py files
#this checks if Python knows it already if not it adds it at the beginning of Python’s import search path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATASET_PATH exists =", DATASET_PATH.exists())
print("OUTPUT_DIR =", OUTPUT_DIR)



Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/Gliner-Work.Dauphine
DATASET_PATH exists = True
OUTPUT_DIR = /content/drive/MyDrive/Gliner-Work.Dauphine/communication_function_outputs


Import libraries

In [3]:
import json
import re
import shlex
import subprocess
import pickle
from dataclasses import asdict, dataclass
from functools import lru_cache

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Settings

In [4]:
#this is the list with expanded number of columns

TEXT_COLUMNS = [
    "Script",
    "Visuel",
    "Titre",
    "MotsClés",
    "Thème",
    "Signature",
]


#this is using a small a amount of columns to see if there are any differeneces

MISTRAL_FOCUS_COLUMNS = [
    "Visuel",
    "Titre",
    "Signature",
    "MotsClés",
]

#this are th context column, they are not main text ad, they will give some useful info about it
METADATA_COLUMNS = [
    "year",
    "Month",
    "Marque",
    "Produit",
    "Variété",
    "Electric",
    "Hybrid",
    "Support",
    "Format",
    "Agence",
    "length",
]

DIMENSIONS = ["informativeness", "expressiveness", "phatic"] #this are the 3 dimensions of Jackobson
#5-point Likert scale
SCORE_MIN = 1
SCORE_MAX = 5


ROUTER_ACCEPT_THRESHOLD = 0.82  #the threshold for accepting the label
ROUTER_REVIEW_THRESHOLD = 0.60 #the threshold for need to review label
ROUTER_HUMAN_THRESHOLD = 0.40 #the threshold for human need label
ROUTER_MAX_AVG_SCORE_DELTA = 0.75 #this keeps teh model average of the disagree below 0.75 between rating of the 3 dimensions
ROUTER_MAX_HARD_DELTA = 2 # this check for teh highest disagreement on a single score
POLICY_PATH = OUTPUT_DIR / "router_policy.pkl" #this is a pickle file = a saved Python object

In [5]:
GEMMA_MODEL = "google/gemma-4-E4B-it" #this is the gemma 4 model i will use please change it if you have a more powerful gpu or a worse one, than T4 GPU
#Gemma_Annotater = "google/gemma-4-31B-it" #strongest version
#Gemma_Annotater = "google/gemma-4-E2B-it" #lightest version
MISTRAL_MODEL = "mistralai/Mistral-Small-4-119B-2603-eagle" #this is the mistral model
#Mistral_Annotater = "mistralai/Mistral-7B-v0.1" #this uses less memory, but I do not recommand it for longer texts as mine
MAX_NEW_TOKENS = 600 #this puts a restriction of 600 tokens for the modelm to avoid long answers

print("Gemma model:", GEMMA_MODEL)
print("Mistral model:", MISTRAL_MODEL)


Gemma model: google/gemma-4-E4B-it
Mistral model: mistralai/Mistral-Small-4-119B-2603-eagle


#This code block is only for debugging scope only
This only keeps the variables

In [6]:
from dataclasses import dataclass, asdict

#I created 3 containers each of them stores main annotations for each ad
@dataclass
class GemmaPrediction:
    row_id: int
    informativeness: int
    expressiveness: int
    phatic: int
    dominant_dimension: str
    dominant_dimension_score: int
    confidence: float
    alternative_dimension: str
    reason: str
    model_name: str

    def to_dict(self):
        return asdict(self)

@dataclass
class MistralDecision:
    row_id: int
    Mistral_informativeness: int
    Mistral_expressiveness: int
    Mistral_phatic: int
    Mistral_dominant_dimension: str
    Mistral_dominant_dimension_score: int
    Mistral_confidence: float
    agrees_with_Gemma: bool
    avg_score_delta: float
    max_score_delta: int
    disagreement_reason: str
    model_name: str

    def to_dict(self):
        return asdict(self)

@dataclass
class RouterDecision:
    row_id: int
    route_action: str
    route_score: float
    route_reason: str

    def to_dict(self):
        return asdict(self)

print("Gemma model:", GEMMA_MODEL)
print("Mistral model:", MISTRAL_MODEL)
print("Max new tokens:", MAX_NEW_TOKENS)


Gemma model: google/gemma-4-E4B-it
Mistral model: mistralai/Mistral-Small-4-119B-2603-eagle
Max new tokens: 600


#Data preprocessing and cleaning

This is useful for the missing values

In [7]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).split()).strip() #okay so we made the output a string, we removed teh spaces and joined them together using one single space and we removed the space from the begining and end

This loads rows from CSV and filter them

In [8]:
def load_rows(limit=None, review_status="needs_label", row_ids=None): #how many rows to keep and we keep only reviews labeled with needs_label
    df = pd.read_csv(DATASET_PATH) #reads csv to pandas DF
    df["row_id"] = pd.to_numeric(df["row_id"], errors="coerce").astype("Int64") #this cleans the row_id column, errors turn the value if invalid in missing one

    if review_status is not None and "Review_Status" in df.columns: #If the first is provided and teh second exists
        df = df[df["Review_Status"] == review_status].copy() #then keep only those values, by default rows with needs_label
#this keeps only the rows whose row_id is in the list of IDs
    if row_ids is not None:
        wanted = {int(x) for x in row_ids}
        df = df[df["row_id"].isin(wanted)].copy()
#this line can be useful if you want to try the labeling of only a subset of entries, for this cxhaneg limit value
    if limit is not None:
        df = df.head(limit).copy()

    return df


This function checks whether your DataFrame contains all the columns you expect and if some are missing, it stops the code with an error listing those missing column names. This can help you avoid mistakes that may need the restart of the code and time and computational costs.

In [9]:
def require_columns(df, columns):
    missing = [c for c in columns if c not in df.columns] #this contains a list with all the columns name we expected that are not present in DF
    if missing:
        raise ValueError(f"Missing columns in dataset: {missing}") #this is the raised error

This takes one row of your dataset and turns the selected columns into one clean block of text.



In [10]:
def build_record_text(row, columns):
    parts = []
    for column in columns:
        text = normalize_text(row.get(column, "")) #if the column exists get its value if not return an empty string
        if text:
            parts.append(f"{column}: {text}") #this creates a dict with var name and its text
    return "\n".join(parts) #this join them together

This code does the same but for metadata text

In [11]:
def build_metadata_text(row):
    parts = []
    for column in METADATA_COLUMNS:
        text = normalize_text(row.get(column, ""))
        if text:
            parts.append(f"{column}: {text}")
    return "\n".join(parts)

This is a quick check

In [12]:
def build_combined_text(row, columns):
    metadata = build_metadata_text(row) # this builds the metadata block from the row.
    record = build_record_text(row, columns=columns) # this builds the main ad-content block from the selected columns.
    if metadata and record:
        return metadata + "\n\n" + record
    return metadata or record #if one is missing return the one thta exists

preview = load_rows(limit=2, review_status="needs_label") #this loads 2 datasets
require_columns(preview, ["row_id"] + TEXT_COLUMNS + METADATA_COLUMNS) #this check if it contains all columns
preview[["row_id", "Marque", "Produit", "Script", "Visuel"]].head()


,row_id,Marque,Produit,Script,Visuel
2100,2100,NISSAN,NISSAN PRESTIGE,"Voix homme : "" NISSAN est fier de vous présent...","Sergio Aguero, nouvel ambassadeur de NISSAN, a..."
2101,2101,VW,VW PRESTIGE,"Voix homme (1) : "" Il n'y a rien de tel que ri...",De la naissance de l'Univers à l'art contempor...


#Annotation rules

the main instruction text for Gemma or Mistral

In [13]:
RUBRIC_TEXT = """You are scoring automotive advertisements with three communication dimensions.

Definitions:
- informativeness: the ad's referential or factual function. It gives product facts, technical details, prices, offers, features, specifications, environmental facts, or concrete real-world information.
- expressiveness: the ad's emotive function. The advertiser expresses feelings, desire, admiration, passion, identity, aspiration, beauty, or an emotional tone.
- phatic: the ad's relationship-maintaining function. The language tries to create or maintain a social bond with the customer, using closeness, togetherness, friendliness, companionship, or affective connection.

Scoring scale:
- 1 = almost absent
- 2 = weak trace
- 3 = clearly present but not dominant
- 4 = strong
- 5 = central and unmistakable

Rules:
- Ads are not zero-sum. An ad can be high on more than one dimension.
- Score each dimension independently from 1 to 5.
- Use text, translated text, visual description, and metadata.
- If two or three dimensions tie for the highest score, use dominant_dimension = mixed.
- dominant_dimension_score must equal the highest of the three scores.
- Keep the reason short and concrete.
"""

This are my supervisor notes please comment this box if you are not in the same situation

In [14]:
SUPERVISOR_NOTES = """Supervisor notes:
- informative content has a social use because it informs people;
- persuasive content is closer to expressiveness;
- referential language points to the real world;
- expressiveness belongs to the initiator and conveys feeling;
- phatic language helps maintain a social relationship and affective bond with the customer;
- many ads contain both informativeness and expressiveness, so score them separately before deriving a dominant one.
"""


This function tells the model what JSON structure should return

In [15]:
def output_schema_text():
    return """Return strict JSON only with:
{
  "informativeness": 1,
  "expressiveness": 1,
  "phatic": 1,
  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",
  "dominant_dimension_score": 1,
  "confidence": 0.0,
  "alternative_dimension": "informativeness|expressiveness|phatic|mixed|",
  "reason": "short explanation"
}
"""

This cleans and standardizes a dimension name

In [16]:
#this function can be helpful if the model uses different words and converts them in your official ones
def normalize_dimension_name(value):
    text = str(value or "").strip().lower() #this cleans the input
    aliases = {
        "informative": "informativeness",
        "information": "informativeness",
        "referential": "informativeness",
        "expressive": "expressiveness",
        "emotive": "expressiveness",
        "emotion": "expressiveness",
        "fatiq": "phatic",
        "fatique": "phatic",
        "fatigue": "phatic",
        "phatique": "phatic",
    }
    text = aliases.get(text, text) #if the word is in aliases replace it, if not keep it
    return text if text in DIMENSIONS or text == "mixed" else "" #this returns one of the official dimensions if it is something else returns ""

This makes sure the score becomes a valid number on your 1 to 5 scale

In [17]:
def clamp_score(value):
    try:
        score = int(round(float(value))) #converts the input into a number from a decimal to a rounded int
    except Exception: #if model gives something invalid it becomes 1
        score = SCORE_MIN
    return max(SCORE_MIN, min(SCORE_MAX, score)) #this forces the output to stay in the 1 to 5 range


This function chooses which of the three dimensions is the dominant one

In [18]:
#this takes a dict and returns a dominant dimension, its score and the second best dimension
def pick_dominant_dimension(scores):
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True) #this sort the dict items by score from highest to lowest
    top_label, top_score = ranked[0] #this takes the highest score and its label
    same_top = [label for label, score in ranked if score == top_score] #this creates a list of all lables that have the same top
    alt = ranked[1][0] if len(ranked) > 1 else "" #this takes the second-ranked label
    if len(same_top) > 1: #if more than one dimension shares the top score, the dominant and second domniant dimensions are mixed
        return "mixed", top_score, alt
    return top_label, top_score, alt

This function takes a raw model output and turns it into a clean, valid prediction dictionary.

In [19]:
def coerce_prediction_dict(payload, row_id, model_name): #the three argumensts are the row output from the model the ID of the row being annotated and the name of the model that produced the answer
    #this cleans the three scores
    scores = {
        "informativeness": clamp_score(payload.get("informativeness")),
        "expressiveness": clamp_score(payload.get("expressiveness")),
        "phatic": clamp_score(payload.get("phatic")),
    }



#this infer the dominant dimension, top score and alternative dimension
    inferred_dom, inferred_score, inferred_alt = pick_dominant_dimension(scores)

#this cleans it
    dominant_dimension = normalize_dimension_name(payload.get("dominant_dimension")) or inferred_dom
    dominant_dimension_score = clamp_score(payload.get("dominant_dimension_score")) #this cleans the domniant score
#this checks if the domniant score actually matches the highest of the three scores
    if dominant_dimension_score != max(scores.values()):
        dominant_dimension_score = inferred_score
#clean the alternative dimension
    alternative_dimension = normalize_dimension_name(payload.get("alternative_dimension")) or inferred_alt
#clean confidence
    try:
        confidence = round(float(payload.get("confidence", 0.5)), 4) #this tries to extract a numeric value if not the it sets to the default of 0.5
    except Exception:
        confidence = 0.5
    confidence = max(0.0, min(1.0, confidence)) #this forces teh ranges from 0.0 to 1.0
#this cleans the reason
    reason = " ".join(str(payload.get("reason", "")).split()).strip() or "model response"

#this returns the final clean dict
    return {
        "row_id": int(row_id),
        "informativeness": scores["informativeness"],
        "expressiveness": scores["expressiveness"],
        "phatic": scores["phatic"],
        "dominant_dimension": dominant_dimension,
        "dominant_dimension_score": dominant_dimension_score,
        "confidence": confidence,
        "alternative_dimension": alternative_dimension,
        "reason": reason,
        "model_name": model_name,
    }


These words increases the informativeness score

In [20]:
INFORMATIVE_KEYWORDS = [
    "offre", "offres", "prix", "euro", "euros", "loyer", "loyers",
    "à partir de", "partir de", "conditions", "condition", "sous condition",
    "garantie", "reprise", "bonus", "prime", "financement", "location",
    "lld", "durée", "entretien", "inclus", "option", "options",
    "autonomie", "recharge", "recharger", "rechargée", "électrique", "electrique",
    "hybride", "batterie", "consommation", "émission", "emission", "co2",
    "technologie", "innovation", "système", "systeme", "connectée", "connecte",
    "navigation", "véhicule", "vehicule", "modèle", "modele", "gamme",
    "réseau", "reseau", "particuliers", "achat", "valable", "ttc", "km"
]

These words increases the expressiveness score

In [21]:
EXPRESSIVE_KEYWORDS = [
    "émotion", "emotion", "passion", "désir", "desir", "rêve", "reve",
    "plaisir", "style", "design", "élégance", "elegance", "beauté", "beaute",
    "séduction", "seduction", "unique", "prestige", "luxe", "liberté", "liberte",
    "joie", "sensation", "chance", "incroyable", "exceptionnel", "audace",
    "charisme", "humour", "étonnant", "etonnant", "surprise", "suspense"
]

These words increases the phatic score

In [22]:
PHATIC_KEYWORDS = [
    "avec vous", "pour vous", "près de vous", "pres de vous", "à vos côtés",
    "a vos cotes", "ensemble", "au quotidien", "quotidien", "toujours",
    "partenaire", "bienvenue", "bonjour", "famille", "partage", "proche",
    "proximité", "proximite", "service", "réseau", "reseau", "nous accompagnons",
    "nous sommes", "pour toute la famille", "à votre service", "a votre service"
]

regex patterns that detect direct address

In [23]:
DIRECT_ADDRESS_PATTERNS = [
    r"\bvous\b", r"\bvotre\b", r"\bvos\b",
    r"\btu\b", r"\btoi\b", r"\bte\b",
    r"\bton\b", r"\bta\b", r"\btes\b",
    r"\bnous\b", r"\bnotre\b", r"\bnos\b",
    r"\bensemble\b"
]

#First try

This function is building a first rough score for one ad using rules instead of a real model.

In [27]:
def baseline_prediction(row, model_name, columns):
    text = build_combined_text(row, columns=columns).lower() #this creates a big text block from the row adn converts it into lowercase, combiens the metadata and selected text columns
    metadata = {key: row.get(key) for key in row.index} #this creates a dict containing all values from the row
#this counts how many times the words from that columns appeared
    info_raw = sum(text.count(k) for k in INFORMATIVE_KEYWORDS)
    expr_raw = sum(text.count(k) for k in EXPRESSIVE_KEYWORDS)
    phat_raw = sum(text.count(k) for k in PHATIC_KEYWORDS)
#this counts how many numbers appeared
    number_hits = len(re.findall(r"\b\d+(?:[.,]\d+)?\b", text))
#this counts exclamation
    exclamation_hits = text.count("!")
#this counts direct addresses
    direct_hits = sum(len(re.findall(pattern, text)) for pattern in DIRECT_ADDRESS_PATTERNS)
    info_raw += min(number_hits, 5) * 0.4 #this adds a bonus to the informativeness up to 5 numbers
    expr_raw += min(exclamation_hits, 3) * 0.5 #this adds a bonus to the expresivenss up to 3 exclamation
    phat_raw += min(direct_hits, 5) * 0.35 #this adds a bonus to the phatic up to 5 direct_hints

#this two are bonuses of 1 if the metadata contains electric and 0.7 if the metadata contains hybrid
    if str(metadata.get("Electric", "")).strip() == "1":
        info_raw += 1.0
    if str(metadata.get("Hybrid", "")).strip() == "1":
        info_raw += 0.7


#This part takes the raw scores you already computed and turns them into the final prediction.
    def raw_to_scale(value):

  #this returns the raw score into 1 to 5 scale
        if value < 0.8:
            return 1
        if value < 2.0:
            return 2
        if value < 3.6:
            return 3
        if value < 5.5:
            return 4
        return 5
#this converts each row score into a clean 1 to 5 score
    scores = {
        "informativeness": raw_to_scale(info_raw),
        "expressiveness": raw_to_scale(expr_raw),
        "phatic": raw_to_scale(phat_raw),
    }
#this uses the scores dict to find the dominant dimension, its score and the alternative
    dominant_dimension, dominant_score, alt = pick_dominant_dimension(scores)

    ranked = sorted([info_raw, expr_raw, phat_raw], reverse=True) #this sorts the raw scores from highest to lowest.
    top = ranked[0] if ranked else 0.0 #this gets teh highest score
    second = ranked[1] if len(ranked) > 1 else 0.0#this gets the sceond highest score
    gap = max(0.0, top - second) #this calculates the difference between them

#this creates a confidence score, which increases when the top is clearly ahead and is the dominant score is high
    confidence = 0.35 + min(gap, 3.0) * 0.12 + dominant_score * 0.05
#if it is a tie the confidence decrease
    if dominant_dimension == "mixed":
        confidence -= 0.12
    confidence = round(max(0.25, min(0.95, confidence)), 4)

#this creates a list for explanation
    reason_bits = []

#if informativeness raw score is sring enough put that explanation, the same for the others
    if info_raw >= 1.5:
        reason_bits.append("factual/technical cues")
    if expr_raw >= 1.5:
        reason_bits.append("emotive/style cues")
    if phat_raw >= 1.2:
        reason_bits.append("bonding/direct-address cues")
    if not reason_bits:
        reason_bits.append("low lexical evidence")
#this is the final prediction output
    return {
        "row_id": int(row["row_id"]),
        "informativeness": scores["informativeness"],
        "expressiveness": scores["expressiveness"],
        "phatic": scores["phatic"],
        "dominant_dimension": dominant_dimension,
        "dominant_dimension_score": dominant_score,
        "confidence": confidence,
        "alternative_dimension": alt,
        "reason": "; ".join(reason_bits),
        "model_name": model_name,
    }

this function tries to find and extract a JSON object from a model’s text output.

In [28]:
def extract_json_object(text):
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL) #this is looking for JSONobject inside the fenced code block
    candidates = [fenced.group(1)] if fenced else [] #if json found add the inside JSON part to the list cadidatesm if not start with an empty list
    candidates.append(text) #this adds the full text as another candidate.

    decoder = json.JSONDecoder() #JSON decoder
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate): #look for every {, since there is the place from where JSON starts
  #We tried to decode it
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip()) #this tries to parse th JSON file
                if isinstance(payload, dict): #if it a dict returns it
                    return payload
            except json.JSONDecodeError: #if it files keep trying
                continue
    raise ValueError("Could not extract JSON from model output.") #if nothing work raise this error

This function loads a text-generation model from transformers and keeps it in memory so it does not need to be reloaded every time.

In [29]:
#this is useful to save time and memory churn
@lru_cache(maxsize=4) # cache decorator, when the cache is full prioritizes removing the least recently used items and it can remember up to 4 different model loads
#this loads teh model
def load_generator(model_id):
    from transformers import pipeline as hf_pipeline #this is the convenient way to load adn run the model
#this loads the generation pipeline
    return hf_pipeline(
        task="text-generation",
        model=model_id,
        tokenizer=model_id,
        trust_remote_code=True, #this lets the HF use costum code and it is necessary for newer models
        device_map="auto", #this lets teh Hf decides wherher to use GPU and CPU
    )


This function runs the model on your prompt

In [30]:
def run_transformers_backend(prompt, model_id):
    generator = load_generator(model_id) #gets the model text-generation pipeline for the model, since load_generator is cached it usually reload the model every time, so this is why we use generator
#this sends the prompt to the model and asks it to generate an answer.
    outputs = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, #this avoids sample randomly
        temperature=0.0, #pushes the model toward a very deterministic answer
        return_full_text=False, #means return only the newly generated answer
    )

#this checks if the model returns nothing, if this is the case the model raise an error
    if not outputs:
        raise RuntimeError("Model returned no output.")
#this cleans it
    return str(outputs[0].get("generated_text", "")).strip()

This function takes the raw text returned by the model and turns it into a clean prediction dictionary.

In [31]:
def parse_model_prediction(raw_text, row_id, model_name):
    payload = extract_json_object(raw_text) #extract JSON from the model output
    return coerce_prediction_dict(payload, row_id=row_id, model_name=model_name) #this cleans and validates it
